##  EXERCICE 4 : Optimisation d'une fonction quadratique mal conditionnée

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.linalg import norm
import sympy as sp

#### QUESTION 1 : Recherche des points critiques (méthode symbolique)

In [ ]:
# Q1 : Points critiques avec SymPy
epsilon = 1e-6
x_sym, y_sym = sp.symbols('x y', real=True)
f_sym = sp.Rational(1, 2) * (epsilon * x_sym**2 + y_sym**2)

# Gradient
# TODO : Complétez le code suivant.
df_dx = sp.diff(...., ....)
df_dy = sp.diff(...., ....)

# Points critiques
# TODO : Complétez le code suivant.
critical_points = sp.solve([.......], [........])
print(f"Points critiques : {critical_points}")

# Hessienne et valeurs propres
# TODO : Complétez le code suivant.
H = sp.Matrix([[...., 0], [....., ...]])
print(f"Hessienne : {H.tolist()}")
print(f"Valeurs propres : {H.eigenvals()}")
print(f"κ = {1/epsilon:.0e}")

#### QUESTION 2 : Méthode du Gradient à Pas Optimal

In [ ]:
# Q2 : Gradient à pas optimal
def gradient_pas_optimal(A, x0, tol=1e-6, max_iter=50000):
    x = x0.copy().astype(float)
    history = [x.copy()]
    grad_norms = []
    for k in range(max_iter):
        grad = A @ x
        grad_norm = norm(grad)
        grad_norms.append(grad_norm)
        if grad_norm < tol:
            break
        alpha = (grad.T @ grad) / (grad.T @ A @ grad)
        x = x - alpha * grad
        history.append(x.copy())
    return x, np.array(history), k + 1, grad_norms

A = np.array([[1e-6, 0], [0, 1]], dtype=float)
x0 = np.array([2.0, 1.0])
tolerances = [1e-2, 1e-4, 1e-6, 1e-8]
results_gpo = {}

for tol in tolerances:
    x_opt, hist, n_iter, gn = gradient_pas_optimal(A, x0, tol=tol)
    results_gpo[tol] = {'x_opt': x_opt, 'history': hist, 'n_iter': n_iter, 'grad_norms': gn}
    print(f"tol={tol:.0e} → {n_iter} itérations, x*=({x_opt[0]:.2e}, {x_opt[1]:.2e})")

#### QUESTION 3 : Courbes de niveau et trajectoire de l'algorithme

In [ ]:
# Q3 : Courbes de niveau et trajectoire
history = results_gpo[1e-6]['history']

x_min, x_max = history[:,0].min(), history[:,0].max()
y_min, y_max = history[:,1].min(), history[:,1].max()
mx, my = max(0.5, abs(x_max-x_min)*0.2), max(0.5, abs(y_max-y_min)*0.2)

x = np.linspace(x_min-mx, x_max+mx, 400)
y = np.linspace(y_min-my, y_max+my, 400)
X, Y = np.meshgrid(x, y)
Z = 0.5 * (1e-6 * X**2 + Y**2)

fig, ax = plt.subplots(figsize=(10, 8))
ax.contour(X, Y, Z, levels=np.logspace(-6, 1, 20), cmap='viridis', alpha=0.7)
ax.plot(history[:,0], history[:,1], 'r.-', markersize=3, label='Trajectoire')
ax.plot(history[0,0], history[0,1], 'go', markersize=10, label='x0')
ax.plot(history[-1,0], history[-1,1], 'ro', markersize=10, label='x*')
ax.set_title('Gradient à Pas Optimal - Courbes de niveau')
ax.legend()
ax.set_aspect('equal')
plt.show()

####  QUESTION 4 : Analyse du nombre d'itérations et observations

In [ ]:
# Q4 : Analyse du nombre d'itérations
eigvals = np.linalg.eigvalsh(A)
kappa = eigvals.max() / eigvals.min()
rho = (kappa - 1) / (kappa + 1)

print(f"κ = {kappa:.0e}, ρ = {rho:.10f}")
print(f"Itérations théoriques ≈ {0.5*kappa*np.log(1e6):.0f}")
print(f"Itérations observées : {results_gpo[1e-6]['n_iter']}")

# Décroissance des résidus
gn = results_gpo[1e-6]['grad_norms']
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(gn)
axes[0].set_title('||∇f|| (linéaire)')
axes[1].semilogy(gn)
axes[1].set_title('||∇f|| (log)')
axes[1].axhline(1e-6, color='r', linestyle='--')
plt.tight_layout()
plt.show()

#### QUESTION 5 : Méthode du Gradient Conjugué (Fletcher-Reeves)

In [ ]:
# Q5 : Gradient Conjugué (Fletcher-Reeves)
def gradient_conjugue(A, x0, tol=1e-6, max_iter=50000):
    x = x0.copy().astype(float)
    history = [x.copy()]
    grad_norms = []
    g = A @ x
    d = -g.copy()
    for k in range(max_iter):
        grad_norm = norm(g)
        grad_norms.append(grad_norm)
        if grad_norm < tol:
            break
        alpha = -(g.T @ d) / (d.T @ A @ d)
        x = x + alpha * d
        history.append(x.copy())
        g_new = A @ x
        beta = (g_new.T @ g_new) / (g.T @ g)
        d = -g_new + beta * d
        g = g_new
    return x, np.array(history), k + 1, grad_norms

results_gc = {}
for tol in tolerances:
    x_opt, hist, n_iter, gn = gradient_conjugue(A, x0, tol=tol)
    results_gc[tol] = {'x_opt': x_opt, 'history': hist, 'n_iter': n_iter, 'grad_norms': gn}
    print(f"tol={tol:.0e} → {n_iter} itérations")

#### QUESTION 5 (suite) : Comparaison détaillée GPO vs GC

In [ ]:
# Q5 (suite) : Comparaison GPO vs GC
print(f"{'Tol':<10} {'GPO':<8} {'GC':<8} {'Ratio':<8}")
for tol in tolerances:
    n_gpo = results_gpo[tol]['n_iter']
    n_gc = results_gc[tol]['n_iter']
    print(f"{tol:<10.0e} {n_gpo:<8} {n_gc:<8} {n_gpo/n_gc:<8.1f}")

# Trajectoires côte à côte
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (hist, title, color) in zip(axes, [
    (results_gpo[1e-6]['history'], f"GPO ({results_gpo[1e-6]['n_iter']} it)", 'r'),
    (results_gc[1e-6]['history'], f"GC ({results_gc[1e-6]['n_iter']} it)", 'b')
]):
    ax.contour(X, Y, Z, levels=np.logspace(-6, 1, 20), cmap='viridis', alpha=0.5)
    ax.plot(hist[:,0], hist[:,1], color+'.-', markersize=3)
    ax.plot(hist[0,0], hist[0,1], 'go', markersize=8)
    ax.plot(hist[-1,0], hist[-1,1], 'ro', markersize=8)
    ax.set_title(title)
    ax.set_aspect('equal')
plt.tight_layout()
plt.show()

# Décroissance comparée
plt.semilogy(results_gpo[1e-6]['grad_norms'], 'r-', label='GPO')
plt.semilogy(results_gc[1e-6]['grad_norms'], 'b-', label='GC')
plt.axhline(1e-6, color='k', linestyle='--', alpha=0.5)
plt.legend()
plt.title('Convergence GPO vs GC')
plt.show()

#### QUESTION 6 : Refaire avec ε = 0.1 (meilleur conditionnement)

In [ ]:
# Q6 : ε = 0.1
A2 = np.array([[0.1, 0], [0, 1]], dtype=float)
eig2 = np.linalg.eigvalsh(A2)
kappa2 = eig2.max() / eig2.min()
print(f"κ(ε=0.1) = {kappa2:.1f} (vs κ(ε=1e-6) = {kappa:.0e})")

print(f"{'Tol':<10} {'GPO':<8} {'GC':<8}")
for tol in tolerances:
    _, _, n_gpo, _ = gradient_pas_optimal(A2, x0, tol=tol)
    _, _, n_gc, _ = gradient_conjugue(A2, x0, tol=tol)
    print(f"{tol:<10.0e} {n_gpo:<8} {n_gc:<8}")

# Trajectoires pour ε=0.1
_, h_gpo, _, _ = gradient_pas_optimal(A2, x0, tol=1e-6)
_, h_gc, _, _ = gradient_conjugue(A2, x0, tol=1e-6)

x2 = np.linspace(-3, 3, 400)
y2 = np.linspace(-2, 2, 400)
X2, Y2 = np.meshgrid(x2, y2)
Z2 = 0.5 * (0.1 * X2**2 + Y2**2)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (hist, title, color) in zip(axes, [
    (h_gpo, f"GPO ε=0.1 ({len(h_gpo)-1} it)", 'r'),
    (h_gc, f"GC ε=0.1 ({len(h_gc)-1} it)", 'b')
]):
    ax.contour(X2, Y2, Z2, levels=20, cmap='viridis', alpha=0.5)
    ax.plot(hist[:,0], hist[:,1], color+'.-', markersize=4)
    ax.plot(hist[0,0], hist[0,1], 'go', markersize=8)
    ax.set_title(title)
    ax.set_aspect('equal')
plt.tight_layout()
plt.show()